# Stacked CNN + Transformer Ensemble on NIH ChestX-ray14

Reruns every experiment in the paper and exports the numbers, LaTeX tables, and figures.

**Before you start**
1. Put `cxr_pipeline.py` and this notebook in **Drive > Colab Notebooks > cxr_experiment**.
2. Runtime → Change runtime type → **A100 GPU** with High-RAM (L4 also works but is slower).
3. Your Drive needs about **8 GB free** (5.6 GB image cache plus checkpoints).

**If Colab disconnects:** rerun cells 1 and 2, then the training cell you were on. Training picks up from the last finished epoch, and models that already finished are skipped.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install timm albumentations kagglehub

import sys, os, glob, importlib
# Finds cxr_pipeline.py wherever you put it in Drive (Colab Notebooks/cxr_experiment by default)
hits = (glob.glob('/content/drive/MyDrive/Colab Notebooks/cxr_experiment/cxr_pipeline.py')
        or glob.glob('/content/drive/MyDrive/Colab Notebooks/**/cxr_pipeline.py', recursive=True)
        or glob.glob('/content/drive/MyDrive/**/cxr_pipeline.py', recursive=True))
assert hits, 'Upload cxr_pipeline.py to Drive > Colab Notebooks > cxr_experiment first'
PROJECT = os.path.dirname(hits[0])
print('Using project folder:', PROJECT)
sys.path.insert(0, PROJECT)
import cxr_pipeline as cx
importlib.reload(cx)

RUN_ROOT = f'{PROJECT}/runs'        # checkpoints, predictions, training histories
OUT_DIR = f'{PROJECT}/results'      # tables, figures, summary.json
LOCAL_CACHE = '/content/cache'      # fast local copy of the image cache
DRIVE_CACHE = f'{PROJECT}/cache'    # persistent copy so you only download once
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Data
The first time, this downloads the dataset from Kaggle (about 42 GB, roughly 30 minutes), splits it by patient using NIH's official
train/test lists, and saves 224×224 copies of every image to Drive. After that it just copies the 5.6 GB cache from Drive.

In [ ]:
cx.get_data(LOCAL_CACHE, DRIVE_CACHE)

## 3. Budget check (optional, about 5 minutes)
Times a few training steps per model on your GPU and estimates hours per run, so you can check your compute units before committing.

In [ ]:
est = cx.probe(cx.MODEL_KEYS, LOCAL_CACHE)

## 4. Main experiment: train the five backbones
Each model trains on its own with early stopping on validation AUC (patience 5). This is the expensive part.
Everything is saved to Drive after every epoch.

In [ ]:
for key in cx.MODEL_KEYS:
    cx.train_model(key, 'main', RUN_ROOT, LOCAL_CACHE)

## 5. Ablations and hyperparameter sweeps
No focal loss, no augmentation, γ ∈ {0, 1, 3}, and learning rates 1e-3 and 5e-5.
By default these run on **DenseNet121 only**, which costs about one extra main run in total.
To run the training ablations on the full ensemble instead, set `ABLATION_MODELS = cx.MODEL_KEYS` (about 2 more full runs).

In [ ]:
SWEEP_MODEL = 'densenet121'
ABLATION_MODELS = [SWEEP_MODEL]

for v in ['gamma0', 'no_aug']:
    for key in ABLATION_MODELS:
        cx.train_model(key, v, RUN_ROOT, LOCAL_CACHE)
for v in ['gamma1', 'gamma3', 'lr1e-3', 'lr5e-5']:
    cx.train_model(SWEEP_MODEL, v, RUN_ROOT, LOCAL_CACHE)

## 6. Evaluate
Stacking, average fusion, CNN-only and Transformer-only ensembles, per-class results, ablation and sweep tables,
and patient-level bootstrap confidence intervals. Runs on CPU in a few minutes. You can run this after step 4 and again after step 5.

In [ ]:
summary = cx.evaluate_all(RUN_ROOT, LOCAL_CACHE, OUT_DIR, sweep_model=SWEEP_MODEL if 'SWEEP_MODEL' in dir() else 'densenet121')

## 7. Deployment benchmark and Grad-CAM

In [ ]:
bench = cx.benchmark(RUN_ROOT, LOCAL_CACHE, OUT_DIR)
cx.gradcam(RUN_ROOT, LOCAL_CACHE, OUT_DIR)

## 8. Package the results
Creates `results.zip` in the same `cxr_experiment` folder. Send that zip back and the paper gets updated from it.

In [ ]:
cx.package(OUT_DIR, RUN_ROOT, f'{PROJECT}/results')